## Sheppard_2020, batch sample, kl1, originalcounts, wasserstein ot-based distances

In [ ]:
import scanpy as sc
import sys
import os
import pandas as pd
import numpy as np


sys.path.append("../../../")
sys.path.append("../../utils/")

import mapqc
import data_cleanup

In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
kl = "1.0"
testrun = False
use_old_counts = True
distance_type = "sinkhorn"

In [ ]:
if testrun:
    suffix = "_test"
    suffix_2 = ""
else:
    suffix = ""
    if use_old_counts == True:
        suffix_2 = "original_counts_"
    else:
        suffix_2 = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_{suffix_2}batch_{batch_variable}_kl{kl}_latent{suffix}.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_counts{suffix}.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix_2}batch_{batch_variable}_kl{kl}{suffix}/mapqc"

In [ ]:
path_results_dir

In [ ]:
path_emb

check if results dir exists, if it's parent directory exists then create the final ("mapqc") directory

In [ ]:
# check if parent directory exists:
parent_dir = os.path.dirname(path_results_dir)
if not os.path.isdir(parent_dir):
    raise ValueError(
        f"Parent directory of the specified results directory ({parent_dir}) does not exist."
    )
else:
    if not os.path.isdir(path_results_dir):
        # create directory:
        print(f"Creating directory {path_results_dir}...")
        os.makedirs(path_results_dir)

In [ ]:
ref_q_key = "r_or_q"
ref_cat = "r"
query_cat = "q"

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)

In [ ]:
adata = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

In [ ]:
del adata_counts

In [ ]:
if (
    "homosapiens_None_2023_None_sikkemalisa_002_d10_1101_2022_03_10_483747"
    in adata.obs.donor_id[0]
):
    print("removing prefix from donor strings")
    adata.obs.donor_id = [
        donor_str.split(
            "homosapiens_None_2023_None_sikkemalisa_002_d10_1101_2022_03_10_483747"
        )[1]
        for donor_str in adata.obs.donor_id
    ]

get center cells from other mapqc results with standard distance metric:

In [ ]:
path_pairwise_dist_results = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/Sheppard_2020/{suffix_2}batch_{batch_variable}_kl{kl}/mapqc/case.pkl"

In [ ]:
path_pairwise_dist_results

In [ ]:
res_pw = mapqc.load(path_pairwise_dist_results)

In [ ]:
center_cell_subset = (
    res_pw.center_cells
)  # try on full set, it seems to be faster than before. Starting on Dec. 3rd, 13:26 or so

run multiple cycles to check variability. Started 1:26pm

In [ ]:
# results_healthy = mapqc.MapqcResults(
#     adata=adata,
#     ref_q_key="r_or_q",
#     ref_cat="r",
#     query_cat="q",
#     condition_key="status",
#     control_cats="control",
#     control_or_case="control",
#     case_cats="IPF",
#     emb_key="X_scanvi_emb",
#     dataset_key="study",
#     sample_key="sample",
#     sampling_group_cat="leiden",
#     seed=0,
#     distance_type=distance_type,
#     center_cells=center_cell_subset,
#     n_cores=4,
#     # n_nhoods=4
# )
# results_healthy.run_mapqc()
results_disease = mapqc.MapqcResults(
    adata=adata,
    ref_q_key="r_or_q",
    ref_cat="r",
    query_cat="q",
    condition_key="status",
    control_cats="control",
    control_or_case="case",
    case_cats="IPF",
    emb_key="X_scanvi_emb",
    dataset_key="study",
    sample_key="sample",
    sampling_group_cat="leiden",
    seed=0,
    distance_type=distance_type,
    center_cells=center_cell_subset,
    # n_nhoods=4,
    n_cores=10,
)

results_disease.run_mapqc()

store results:

In [ ]:
if distance_type == "pairwise_euclidean":
    # results_healthy.save(os.path.join(path_results_dir, "control.pkl"))
    results_disease.save(os.path.join(path_results_dir, "case.pkl"))
elif distance_type == "centroid_euclidean":
    # results_healthy.save(os.path.join(path_results_dir, "control_centroid.pkl"))
    results_disease.save(os.path.join(path_results_dir, "case_centroid.pkl"))
elif distance_type == "sinkhorn":
    # results_healthy.save(os.path.join(path_results_dir, "control_sinkhorn.pkl"))
    results_disease.save(os.path.join(path_results_dir, "case_sinkhorn.pkl"))
else:
    raise ValueError("Unknown distance type")

load results if starting from here:

In [ ]:
# if distance_type == "pairwise_euclidean":
#     results_healthy = mapqc.load(os.path.join(path_results_dir, "control.pkl"))
#     results_disease = mapqc.load(os.path.join(path_results_dir, "case.pkl"))
# elif distance_type == "centroid_euclidean":
#     results_healthy = mapqc.load(os.path.join(path_results_dir, "control_centroid.pkl"))
#     results_disease = mapqc.load(os.path.join(path_results_dir, "case_centroid.pkl"))
# elif distance_type == "sinkhorn":
#     results_healthy = mapqc.load(os.path.join(path_results_dir, "control_sinkhorn.pkl"))
#     results_disease = mapqc.load(os.path.join(path_results_dir, "case_sinkhorn.pkl"))
# else:
#     raise ValueError("Unknown distance type")

In [ ]:
results_healthy.result_stats

In [ ]:
results_disease.result_stats

In [ ]:
cmap = {"r control": "grey", "q control": "dodgerblue", "q IPF": "midnightblue"}

In [ ]:
# create variable splitting data in ref control, query control
adata.obs["r_q_status"] = [
    f"{r_q} {stat}" for r_q, stat in zip(adata.obs.r_or_q, adata.obs.status)
]

In [ ]:
sc.pl.umap(
    adata,
    color="r_q_status",
    palette=cmap,
    frameon=False,
)

### Results healthy

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, color_by="mean_pval_corr", size=0.5)

In [ ]:
sc.pl.umap(adata, color=["study"], frameon=False)

In [ ]:
mapqc.pl.umap_mapqc_scores(
    results_healthy, color_by="mean_pval_corr_significant", size=0.5
)

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, color_by="mapqc_score>2", size=0.5)

In [ ]:
results_healthy.adata

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(
    results_healthy, label_x_ticks_by="ann_level_4_pred"
)

#### intermezzo: look into specific neighborhood

In [ ]:
nhood_oi = results_healthy.nhood_stats.index[
    results_healthy.nhood_stats.nhood_number == 452
][0]

In [ ]:
results_healthy.nhood_stats.loc[nhood_oi, :]

In [ ]:
mapqc.pl.umap_center_cell(results_healthy, neighborhood_center_cell=nhood_oi)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(results_healthy, neighborhood=nhood_oi)

In [ ]:
mapqc.pl.umap_show_neighborhood(
    results_healthy,
    neighborhood=nhood_oi,
    color_by="study",
)

In [ ]:
mapqc.pl.umap_q_ref_sample_shaded(results_healthy, neighborhood=nhood_oi, dotsize=10)

In [ ]:
mapqc.pl.umap_neighborhood(
    results_healthy,
    color_by=["sample", "study", "mapqc_score", "mapqc_score>2"],
    neighborhood=nhood_oi,
    recalculate_umap=False,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
)

In [ ]:
mapqc.pl.umap_neighborhood(
    results_healthy,
    color_by=["sample", "study", "mapqc_score>2"],
    neighborhood=nhood_oi,
    recalculate_umap=True,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
)

check why mapqc score is so low for cells from distant sample in this neighborhood

In [ ]:
# get q samples in nhood of interest
results_healthy.sample_stats_per_nhood_df.loc[
    (results_healthy.sample_stats_per_nhood_df.nhood == nhood_oi)
    & (results_healthy.sample_stats_per_nhood_df.sample_r_or_q == "q"),
    :,
]

Note that the NML1_All sample does have a high mapqc score, as it should. now get a cell from this nhood, and check which other nhood mapqc scores it has...

#### end of intermezzo

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, size=0.5)

In [ ]:
mapqc.pl.umap_filtering_info(results_healthy)

In [ ]:
adata.obs["donor_id_q"] = None
adata.obs.loc[adata.obs.r_or_q == "q", "donor_id_q"] = adata.obs.loc[
    adata.obs.r_or_q == "q", "donor_id"
].tolist()

In [ ]:
sc.pl.umap(
    adata,
    color=["study", "donor_id_q", "ann_level_3_pred", "ann_level_3"],
    frameon=False,
    wspace=0.7,
    ncols=2,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(results_healthy, box_category="sample")

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy, box_category="ann_level_2_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy,
    box_category="ann_level_2_pred",
    color_category="donor_id",
    color_high_median_boxes_red=False,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy, box_category="ann_level_3_pred", color_category="donor_id"
)

In [ ]:
custom_palette = {"control": "dodgerblue", "IPF": "midnightblue"}

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="ann_level_2_pred",
    group_category="donor_id",
    # palette=custom_palette,
    min_cells_per_cat=50,
    vmax=10,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="ann_level_3_pred",
    group_category="status",
    palette=custom_palette,
    min_cells_per_cat=50,
    vmax=10,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="ann_level_4_pred",
    group_category="status",
    palette=custom_palette,
    min_cells_per_cat=50,
    vmax=10,
)

### Results disease

In [ ]:
mapqc.pl.umap_mapqc_scores(results_disease, size=1)

In [ ]:
mapqc.pl.umap_filtering_info(results_disease)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(results_disease)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(results_disease, box_category="sample")

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="ann_level_2_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="ann_level_3_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="ann_level_4_pred", color_category="donor_id"
)